## Numpy
#### Создание массивов


In [ ]:
import numpy as np

# Из списков Python
a = np.array([1.0, 2.0, 3.0, 4.0])
print(a.dtype)   # float64
print(a.shape)   # (4,) — одномерный, 4 элемента

# Двумерный массив (матрица)
m = np.array([[1, 2, 3], [4, 5, 6]])
print(m.shape)   # (2, 3) — 2 строки, 3 столбца

# Фабричные функции
zeros    = np.zeros((3, 4), dtype=np.float32)
ones     = np.ones((2, 2))
eye      = np.eye(3)                   # единичная матрица
arange   = np.arange(0, 10, 2)        # [0, 2, 4, 6, 8]
linspace = np.linspace(0, 1, 5)       # [0.0, 0.25, 0.5, 0.75, 1.0]
random   = np.random.randn(100, 5)    # 100 наблюдений, 5 признаков, стандартное нормальное

#### dtype — почему это важно

In [ ]:
# Сравнение памяти
big_float64 = np.zeros(1_000_000, dtype=np.float64)
big_float32 = np.zeros(1_000_000, dtype=np.float32)
print(big_float64.nbytes)   # 8 000 000 байт (8 МБ)
print(big_float32.nbytes)   # 4 000 000 байт (4 МБ)

# Целочисленные типы
arr_int32 = np.array([1, 2, 3], dtype=np.int32)
arr_int64 = np.array([1, 2, 3], dtype=np.int64)

# Практические правила для экономических данных:
# float64 → цены, доходности, значения ВВП (по умолчанию, наиболее безопасный)
# float32 → матрицы признаков для ML, когда важна память
# int32   → счётчики, индексы
# bool    → маски

#### Изменение формы

In [ ]:
flat   = np.arange(12)           # [ 0  1  2 ... 11]
matrix = flat.reshape(3, 4)      # матрица 3×4
column = flat.reshape(-1, 1)     # вектор-столбец 12×1 (-1 = вывести автоматически)
row    = flat.reshape(1, -1)     # вектор-строка 1×12

#### Срезы

In [ ]:
m = np.array([[10, 20, 30],
              [40, 50, 60],
              [70, 80, 90]])

print(m[0, :])      # [10 20 30]        — первая строка
print(m[:, 1])      # [20 50 80]        — второй столбец
print(m[1:3, 0:2])  # [[40 50] [70 80]] — подматрица

#### Булево индексирование

In [ ]:
prices = np.array([100.5, 85.0, 200.0, 50.0, 175.5])

# Создать булеву маску
expensive = prices > 100
print(expensive)          # [True False True False True]

# Применить маску
print(prices[expensive])  # [100.5 200.0 175.5]

# Комбинированные условия
print(prices[(prices > 80) & (prices < 180)])  # [100.5 85.0 175.5]

# Замена значений через where
clipped = np.where(prices > 150, 150, prices)
print(clipped)  # [100.5  85.   150.    50.   150. ]

#### Представление (view) vs Копия (copy) — критически важно для производительности и корректности

In [ ]:
a = np.arange(10)

# Срезы — это ПРЕДСТАВЛЕНИЯ, они разделяют память с исходным массивом
b = a[2:5]
b[0] = 999
print(a)   # [  0   1 999   3   4   5   6   7   8   9]  ← исходный изменён!

# Расширенное индексирование возвращает КОПИЮ
c = a[[0, 3, 6]]
c[0] = -1
print(a)   # не изменился

# Принудительное копирование
d = a[2:5].copy()
d[0] = 0
print(a)   # не изменился

#### Broadcasting — правила

In [ ]:
# Правило: размерности совместимы, если они равны ИЛИ одна из них равна 1
# NumPy расширяет ось с размерностью 1 до совпадения с другой

# Скалярное broadcasting
a = np.array([1, 2, 3])
print(a * 10)   # [10 20 30]

# 1D + 2D
row_vector = np.array([1, 2, 3])    # форма (3,)
matrix = np.ones((4, 3))            # форма (4, 3)
result = matrix + row_vector        # форма (4, 3) — строка добавляется к каждой строке
print(result)

# Вектор-столбец + вектор-строка → 2D внешнее произведение
col = np.array([[1], [2], [3]])     # форма (3, 1)
row = np.array([10, 20, 30])        # форма (3,)
outer = col + row                   # форма (3, 3)
print(outer)
# [[11 21 31]
#  [12 22 32]
#  [13 23 33]]

#### Почему циклы медленные — накладные расходы интерпретатора

In [ ]:
# Циклы Python: каждая итерация включает:
# 1. Проверку типов
# 2. Выделение памяти для объекта Python
# 3. Диспетчеризацию байткода
# NumPy: вызывает оптимизированные процедуры C/Fortran на непрерывной памяти

# Демонстрация с %timeit (запускать в Jupyter)
import numpy as np

N = 1_000_000
data = np.random.randn(N)

# Метод 1: цикл Python
def normalize_loop(arr):
    mean = sum(arr) / len(arr)
    std = (sum((x - mean)**2 for x in arr) / len(arr)) ** 0.5
    return [(x - mean) / std for x in arr]

# Метод 2: списочное включение
def normalize_listcomp(arr):
    lst = arr.tolist()
    mean = sum(lst) / len(lst)
    std = (sum((x - mean)**2 for x in lst) / len(lst)) ** 0.5
    return [(x - mean) / std for x in lst]

# Метод 3: NumPy векторизованный (z-score нормализация)
def normalize_numpy(arr):
    return (arr - arr.mean()) / arr.std()

# Типичные результаты на 1M элементов:
# loop         ~3000 мс
# listcomp     ~300 мс
# numpy        ~5 мс  (в 600 раз быстрее цикла)

#### Справочник векторизованных операций

In [ ]:
arr = np.random.randn(1000)

# Агрегации
print(arr.mean(), arr.std(), arr.min(), arr.max())
print(arr.sum(), np.median(arr))

# По осям (2D)
m = np.random.randn(100, 5)
col_means = m.mean(axis=0)   # среднее по каждому столбцу (форма: 5,)
row_sums  = m.sum(axis=1)    # сумма по каждой строке (форма: 100,)

# Поэлементные операции
returns = np.array([0.05, -0.02, 0.08, -0.01, 0.03])
cumulative = np.cumprod(1 + returns) - 1  # накопленная доходность
print(cumulative)

# Булевы агрегации
print((arr > 0).sum())    # количество положительных значений
print((arr > 0).mean())   # доля положительных

#### `np.where` — векторизованный if/else

In [ ]:
ratings = np.array([3.2, 4.8, 2.1, 4.1, 1.5, 3.9])

# Присвоить оценку по порогу
grades = np.where(ratings >= 4.0, "A",
         np.where(ratings >= 3.0, "B", "C"))
print(grades)  # ['B' 'A' 'C' 'A' 'C' 'B']

### Бенчмарк производительности

In [ ]:
import numpy as np
import time

N = 1_000_000
data = np.random.randn(N)

def time_it(label, func, *args, reps=3):
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        result = func(*args)
        times.append(time.perf_counter() - t0)
    avg = sum(times) / len(times)
    print(f"{label:30s}: {avg*1000:.1f}мс")
    return result

# --- Z-score нормализация ---
def loop_normalize(arr):
    arr_list = arr.tolist()
    mean = sum(arr_list) / len(arr_list)
    std = (sum((x - mean) ** 2 for x in arr_list) / len(arr_list)) ** 0.5
    return [(x - mean) / std for x in arr_list]

def listcomp_normalize(arr):
    arr_list = arr.tolist()
    mean = sum(arr_list) / len(arr_list)
    std = (sum((x - mean) ** 2 for x in arr_list) / len(arr_list)) ** 0.5
    return [(x - mean) / std for x in arr_list]

def numpy_normalize(arr):
    return (arr - arr.mean()) / arr.std()

# Запуск бенчмарков
time_it("Цикл Python", loop_normalize, data)
time_it("Списочное включение", listcomp_normalize, data)
time_it("NumPy векторизованный", numpy_normalize, data)

# Бонус: визуализация ускорения
import matplotlib.pyplot as plt
labels = ["Цикл Python", "Список. включ.", "NumPy"]
# подставить реальные значения из результатов выше
times_ms = [3000, 300, 5]   # приблизительно
plt.bar(labels, times_ms, color=["#d62728", "#ff7f0e", "#2ca02c"])
plt.ylabel("Время (мс)")
plt.title(f"Нормализация {N:,} элементов")
plt.yscale("log")
plt.tight_layout()
plt.savefig("benchmark.png", dpi=150)
plt.show()